In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 1.2 The Damped, Driven Pendulum

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume I — Elementary Mechanics",
    number="1.2",
    title="The Damped, Driven Pendulum",
    blurb="From a linear resonance curve to the period-doubling road to "
    "chaos — one nonlinear oscillator, three regimes.",
    difficulty="intermediate",
    estimate="60–90 min",
)

## Notebook overview

The pendulum is the first oscillator every student meets, and in the
small-angle limit it is utterly tame: a single sine wave. Add damping and a
periodic push, then let the swing grow large enough that $\sin\theta$ can no
longer be replaced by $\theta$, and the *same* one-degree-of-freedom system
becomes one of the cleanest textbook routes to **chaos**. It is the simplest
place to watch a period-doubling cascade unfold.

We will (1) implement the equations of motion, (2) recover the simple-harmonic
period in the small-angle limit, (3) measure the damped decay rate and shifted
frequency, (4) reproduce the linear resonance curve from a driven steady
state, (5) locate the resonance peak, (6) build a bifurcation diagram showing
the period-doubling cascade, (7) confirm chaos at the canonical parameters
by measuring a positive Lyapunov exponent, and (8) animate the damped free
decay. Two animations punctuate the notebook: a worked one (the chaotic bob
swinging on its attractor) in Exercise 7, and one you build in Exercise 8.

> **How to read the checks.** Each exercise ends with a validation that
> compares our result to an expected physical fact. A ✗ does *not* by itself
> mean the answer is wrong: it means the output didn't match what the check
> expected, which may be a real error, a different-but-valid convention (a
> sign, a unit, an array order), or simply too tight a tolerance. Treat a ✗ as
> a prompt to locate the discrepancy; passing is strong evidence of
> correctness, not proof.

> **Scope.** This is a working review, not a textbook chapter. For the
> mechanics of the pendulum see Nolting, *Theoretical Physics 1*
> {cite}`nolting1`; for the bifurcation and chaos picture, Strogatz,
> *Nonlinear Dynamics and Chaos* {cite}`strogatz`.

## Theory in brief

### Equation of motion and nondimensionalisation

A pendulum of mass $m$ on a rod of length $\ell$, with a linear (viscous)
damping torque $-b\dot\theta$ and a sinusoidal drive torque
$\tau_0\cos(\Omega t)$, obeys the torque balance

$$
m\ell^2 \ddot\theta = -b\,\dot\theta - m g \ell \sin\theta
                      + \tau_0 \cos(\Omega t).
$$

Divide by $m\ell^2$ and introduce the natural frequency
$\omega_0 = \sqrt{g/\ell}$. Measuring time in units of $1/\omega_0$ (so the
dimensionless time is $\omega_0 t$, and frequencies are measured in units of
$\omega_0$) collapses every system onto the **Baker–Gollub** form

```{math}
:label: eq-bg
\ddot\theta + \frac{1}{q}\,\dot\theta + \sin\theta = A\cos(\omega_d t)
```

with just three numbers: the **quality factor** $q$ (the damping rate is
$1/q$), the dimensionless **drive amplitude** $A$, and the **drive frequency**
$\omega_d = \Omega/\omega_0$. We carry the state $\mathbf s = (\theta, \omega)$
with $\omega \equiv \dot\theta$, so the second-order ODE becomes the first-order
system $\dot\theta = \omega$, $\dot\omega = -\omega/q - \sin\theta + A\cos(\omega_d t)$.

### Small-angle linear limit

For $|\theta|\ll 1$, $\sin\theta \approx \theta$ and {eq}`eq-bg` becomes the
**damped, driven linear oscillator**

```{math}
:label: eq-linosc
\ddot\theta + \tfrac1q\dot\theta + \theta = A\cos(\omega_d t),
```

which we can solve exactly, giving us three independent checks on the
integrator (Taylor, *Classical Mechanics*, Ch. 5, carries the solution of the
damped, driven linear oscillator out in full; we quote its three results).

**Free, undamped** ($A=0$, $q\to\infty$): {eq}`eq-linosc` is simple harmonic
motion at $\omega_0 = 1$, i.e. period $2\pi$.

**Free, underdamped** ($A=0$): the amplitude decays under the envelope
$e^{-t/(2q)}$ while oscillating at the shifted frequency

```{math}
:label: eq-omega1
\omega_1 = \sqrt{1 - \tfrac{1}{4q^2}}.
```

**Driven steady state**: after transients die, the response is sinusoidal at
the drive frequency with amplitude

```{math}
:label: eq-resonance
\Theta(\omega_d) = \frac{A}{\sqrt{(1-\omega_d^2)^2 + (\omega_d/q)^2}},
```

a resonance curve that peaks at

```{math}
:label: eq-peak
\omega_d^\star = \sqrt{1 - \tfrac{1}{2q^2}}.
```

### The nonlinear regime

Once $A$ is large enough to drive the pendulum to wide swings, the $\sin\theta$
nonlinearity matters and the tidy linear picture fails. Holding $q$ and
$\omega_d$ fixed and turning $A$ up, the steady response loses stability in a
**period-doubling cascade**: a period-1 orbit (closing after one drive period)
gives way to period-2, then period-4, 8, …, accumulating at a critical drive
beyond which the motion is **chaotic**: bounded, deterministic, yet sensitive
to initial conditions. The canonical chaotic parameters used throughout the
literature are
$$
q = 2, \qquad A = 1.5, \qquad \omega_d = \tfrac{2}{3}.
$$

### The physical setup

A bob of mass $m$ hangs from a pivot on a rigid rod of length $\ell$, swinging
to an angle $\theta$ from the downward vertical. Two extra torques act on top of
gravity: a viscous **damping** $-b\dot\theta$ that opposes the motion, and an
external sinusoidal **drive** $\tau_0\cos(\Omega t)$. Those three ingredients
(restoring gravity, damping, periodic forcing) are the whole of {eq}`eq-bg`.

In [ ]:
# (solution hidden on the public site)


---
## Setup

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

from ecp import validate
from ecp.animate import show

rng = np.random.default_rng(0)

# Dimensionless Baker–Gollub convention (omega_0 = 1) throughout:
#     theta'' + (1/q) theta' + sin(theta) = A cos(omega_d t),
# with state s = (theta, omega), omega = theta'. The quality factor q sets the
# damping (rate 1/q); A is the drive amplitude; WD is the drive frequency.
# Canonical chaotic parameters, used in Exercises 6–7:
Q_CHAOS = 2.0
A_CHAOS = 1.5
WD_CHAOS = 2.0 / 3.0

## Exercise 1 — Implement the equations of motion

Everything in this notebook runs on the one second-order equation {eq}`eq-bg`.
A solver needs it as a first-order system, so we carry the state $\mathbf s =
(\theta, \omega)$ with $\omega \equiv \dot\theta$, and carrying $q$, $A$,
$\omega_d$ as arguments lets every later exercise select its regime by passing
different numbers.

Write the right-hand side `rhs(t, s, q, A, wd)` that returns
$(\dot\theta, \dot\omega)$ for {eq}`eq-bg`.

1. Unpack $\mathbf s = (\theta, \omega)$; the first derivative is just
   $\dot\theta = \omega$.
2. Return $\dot\omega = -\omega/q - \sin\theta + A\cos(\omega_d t)$, reading
   the three terms straight off {eq}`eq-bg`.

**Write this one yourself** — the implementation is the lesson.

In [ ]:
# (solution hidden on the public site)


### Validation 1 — the small-angle restoring force

With the drive and damping switched off, a tiny displacement must feel the
linear restoring acceleration $-\omega_0^2\theta = -\theta$ (recall
$\omega_0=1$). Evaluate the angular acceleration at $\theta=10^{-3}$.

In [ ]:
accel = rhs(0.0, [1e-3, 0.0], 1e12, 0.0, 0.0)[1]
validate.close(accel, -1e-3, "small-angle restoring force is -ω₀²θ", rtol=1e-6)

## Exercise 2 — Small-angle free oscillation recovers the SHM period

The first check on the integrator is the tamest limit of {eq}`eq-linosc`: with
no drive and negligible damping, a small swing is simple harmonic motion at
$\omega_0 = 1$, period $2\pi$. If the code can't reproduce that, nothing
downstream is trustworthy.

1. With `scipy.integrate.solve_ivp` (`DOP853`, `rtol=1e-10`, `atol=1e-12`) on a
   dense `t_eval`, integrate the undriven ($A=0$), essentially undamped ($q$
   enormous) pendulum from a small angle.
2. Measure the period from successive zero crossings of $\theta(t)$ (the
   `zero_crossing_times` helper) and confirm it equals $2\pi/\omega_0 = 2\pi$.
   (Note the densely sampled `t_eval`: a sparse adaptive solution drawn with
   straight segments looks jagged even when the numbers are right.)

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(T_measured, 2 * np.pi, "small-angle period is 2π/ω₀", rtol=2e-3)

## Exercise 3 — Damped free decay: rate and frequency

Turn damping on (still no drive) and {eq}`eq-linosc` predicts two things at
once: the amplitude decays under the envelope $e^{-t/(2q)}$, and the
oscillation frequency drops to $\omega_1$ from {eq}`eq-omega1`. Both are
measurable from a single free-decay run.

Set $q=5$, $A=0$ and integrate from a small angle with
`scipy.integrate.solve_ivp` (`DOP853`) on a dense `t_eval`.

1. Measure the decay rate from the upper envelope of the peaks (the
   `refined_maxima` helper, then `numpy.polyfit` on the log) and compare it to
   $1/(2q)$.
2. Measure the frequency from the zero crossings and compare it to $\omega_1$ of
   {eq}`eq-omega1`. Overlay the analytic envelope on $\theta(t)$.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(decay_rate, 1 / (2 * q3), "envelope decays at 1/(2q)", rtol=1e-3)
validate.close(
    freq, np.sqrt(1 - 1 / (4 * q3**2)), "damped frequency is √(1-1/4q²)", rtol=1e-3
)

## Exercise 4 — Driven resonance curve (linear regime)

Now switch the drive on but keep it gentle, so {eq}`eq-linosc` still holds. Its
steady-state solution is the resonance curve {eq}`eq-resonance`: the last of
the three exact linear results, and the one that needs a real
transient-then-measure protocol to test.

1. Drive the pendulum at $A=0.02$ (firmly linear) and $q=5$ and sweep
   $\omega_d$ across the resonance.
2. For each drive frequency, integrate away the transient and measure the
   steady-state amplitude over the last few drive periods (the
   `steady_amplitude` helper).
3. Confirm the sweep traces {eq}`eq-resonance`.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# Compare a few points away from the peak, where linear theory is sharpest.
check_idx = [4, 12, 36]  # wd = 0.5, 0.7, 1.3 on the grid
validate.close(
    amp_num[check_idx],
    amp_curve[check_idx],
    "steady-state amplitude matches the linear resonance curve",
    rtol=2e-2,
)

## Exercise 5 — Resonance peak location

The resonance curve {eq}`eq-resonance` does not peak at the natural frequency:
damping pulls the maximum down to $\omega_d^\star$ given by {eq}`eq-peak`. The
sweep from Exercise 4 already contains the data to test this.

1. The Exercise-4 grid is too coarse to resolve the small (~1%) shift, so
   refine the sweep near the peak and find the drive frequency that maximises
   the response (`numpy.argmax`).
2. Confirm it sits at $\omega_d^\star$ of {eq}`eq-peak`, slightly below the
   natural frequency.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(wd_peak_num, wd_star, "resonance peaks at √(1-1/2q²)", atol=0.05)

## Exercise 6 — Period-doubling cascade (bifurcation diagram)

Now leave the linear world behind: drive {eq}`eq-bg` hard enough that
$\sin\theta$ can no longer be linearised, and the tidy resonance picture
{eq}`eq-resonance` fails. The route the system takes to chaos is a
period-doubling cascade, and a stroboscopic (Poincaré) sample makes it visible.

1. Fix $q=2$, $\omega_d=2/3$ and sweep the drive amplitude $A$ across
   $[1.35, 1.5]$.
2. For each $A$, integrate past the transient (`scipy.integrate.solve_ivp`) and
   **strobe** the motion: record the state once per drive period (the `strobe`
   helper — a Poincaré sample).
3. Plot the strobed angle against $A$ and read off the cascade: one band splits
   to two, then four, then dissolves into the broad smear of chaos.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# Structural check: the response is more complex (wider strobe spread) at the
# chaotic drive than at the periodic one. Measure spread on the strobe velocity,
# which (unlike the angle) never wraps.
_, w_lo = strobe(1.35)
_, w_hi = strobe(1.50)
spread_lo, spread_hi = np.ptp(w_lo), np.ptp(w_hi)
validate.check(
    spread_hi > spread_lo,
    "response is more complex at higher drive",
    f"strobe spread: A=1.5 → {spread_hi:.3f}  vs  A=1.35 → {spread_lo:.3f}",
)

## Exercise 7 — Chaos: sensitive dependence at the canonical parameters

At the canonical chaotic parameters $q=2$, $A=1.5$, $\omega_d=2/3$, {eq}`eq-bg`
has no periodic steady state at all: the motion is chaotic. The formal
signature is sensitive dependence: two nearby starts separate exponentially,
$\delta(t)\sim\delta_0 e^{\lambda t}$, with a positive largest **Lyapunov
exponent** $\lambda$.

1. Settle a reference trajectory onto the attractor with
   `scipy.integrate.solve_ivp`, then launch a second differing by
   $\delta_0 = 10^{-8}$ in angle.
2. Fit the slope of $\ln\delta(t)$ in its exponential-growth window
   (`numpy.polyfit`) to estimate $\lambda$.
3. Animate the chaotic bob (worked below) and view the strange attractor as a
   Poincaré section; the final check confirms $\lambda > 0$.

In [ ]:
# (solution hidden on the public site)


A Poincaré section (the strobe points of a single long chaotic run) fills
out the **strange attractor** the trajectories are confined to.

In [ ]:
# (solution hidden on the public site)


### A worked animation — the chaotic bob

The Poincaré section is the chaos seen *stroboscopically*; the animation is the
chaos seen *continuously*. We carry the already-settled `base` state forward a
few dozen drive periods, map the angle to the physical bob position
$(x,y) = (\sin\theta, -\cos\theta)$, and watch the rod swing, loop over the top,
and never repeat. This is the worked example for the two-animation rule; you
build the second one in Exercise 8.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.positive(lyap_slope, "largest Lyapunov exponent is positive (chaotic)")

## Exercise 8 — Animate the damped free decay

This is the second animation, and it is yours to build. We return to the
linear-limit physics of Exercise 3: an undriven, underdamped swing whose
amplitude decays under the envelope $e^{-t/(2q)}$. Here you *animate* $\theta(t)$
riding inside that envelope, and the validation checks the physics carried by
the animated data (the decay rate) against the $1/(2q)$ of {eq}`eq-linosc`.

1. Integrate the undriven pendulum at $q=4$ from $\theta_0 = 0.2$ with
   `scipy.integrate.solve_ivp` (`DOP853`) on a fine, dense `t_eval`, long enough
   for several oscillations to decay visibly.
2. Animate $\theta(t)$ advancing under the dashed envelope
   $\pm\theta_0 e^{-t/(2q)}$ with `FuncAnimation` (subsample to keep the frame
   count light).
3. Measure the decay rate from the peaks of the animated trace; the validation
   compares it to $1/(2q)$.

In [ ]:
# (solution hidden on the public site)


### Validation 8 — the animated decay falls at rate 1/(2q)

The envelope of the animated $\theta(t)$ must shrink at the rate $1/(2q)$ set by
{eq}`eq-linosc`. The check reads the decay rate fitted from the animated trace's
peaks, so it validates the physics on screen rather than the `FuncAnimation`
object.

In [ ]:
validate.close(
    decay_rate8,
    1 / (2 * q8),
    "animated decay falls at rate 1/(2q)",
    rtol=2e-3,
)

```{admonition} With your assistant
:class: tip
The animation scaffolding of Exercise 8 — figure sizing, `FuncAnimation`
wiring, axis dressing — is exactly the kind of code to delegate outright: ask
your assistant for it and spend the time on the physics. The check stays
yours, and it is physical: the animated decay must halve its amplitude on the
timescale your Exercise 3 fit measured.
```

## Notebook summary

- The pendulum equation integrated across regimes: the small-angle period $2\pi/\omega_0$,
  damped free decay with envelope $\sim e^{-t/2q}$, and the driven resonance peak at
  $\omega=\omega_0\sqrt{1-1/2q^2}$.
- The route to chaos: a period-doubling cascade (bifurcation diagram) and sensitive dependence
  at the canonical parameters, confirmed by a positive largest Lyapunov exponent.

## Outlook

- **Basins of attraction.** In the multistable (non-chaotic) driven regime,
  different initial conditions settle onto different periodic attractors. Map
  which start ends up where and colour the basins.
- **The full bifurcation diagram.** Extend the $A$-sweep over a wider range,
  resolve several period-doublings, and estimate the **Feigenbaum ratio** of
  successive bifurcation spacings.
- **Integrator choice.** Compare RK45 against a symplectic-with-forcing scheme
  on very long stroboscopic runs: does the attractor's fine structure survive?

### References

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()